## Setup

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pyomo.environ as pyo
import z3
from pyomo.repn import generate_standard_repn

DATA_DIR = Path("stock_data")
TRADING_DAYS_PER_YEAR = 252  # used to annualize daily mean/covariance stats

### Data loading

In [2]:
def load_prices() -> pd.DataFrame:
    """Long-format aligned adjusted close: one row per (Date, ticker)."""
    wide = pd.read_csv(DATA_DIR / "aligned_adj_close.csv", index_col="Date", parse_dates=True)
    # long format makes it easy to merge in per-ticker metadata (sector, name, ...) below
    long = wide.melt(ignore_index=False, var_name="ticker", value_name="adj_close").reset_index()
    return long


def load_metadata() -> pd.DataFrame:
    return pd.read_csv(DATA_DIR / "metadata.csv")


def load_prices_with_metadata() -> pd.DataFrame:
    prices = load_prices()
    metadata = load_metadata()
    return prices.merge(metadata, on="ticker", how="left")

### Portfolio optimization (Markowitz minimum-variance)

In [3]:
def compute_annualized_stats() -> tuple[pd.Series, pd.DataFrame]:
    """Annualized mean returns and covariance matrix, indexed by ticker."""
    wide = pd.read_csv(DATA_DIR / "aligned_adj_close.csv", index_col="Date", parse_dates=True)
    daily_returns = wide.pct_change().dropna()
    mean_returns = daily_returns.mean() * TRADING_DAYS_PER_YEAR
    cov_matrix = daily_returns.cov() * TRADING_DAYS_PER_YEAR
    return mean_returns, cov_matrix


def build_markowitz_model(target_return: float) -> pyo.ConcreteModel:
    """Long-only, fully-invested minimum-variance model for a given target return (unsolved)."""
    mean_returns, cov_matrix = compute_annualized_stats()
    tickers = list(mean_returns.index)

    model = pyo.ConcreteModel()
    model.tickers = pyo.Set(initialize=tickers)
    model.weight = pyo.Var(model.tickers, bounds=(0, 1))

    model.budget = pyo.Constraint(expr=sum(model.weight[t] for t in tickers) == 1)
    model.target = pyo.Constraint(
        expr=sum(model.weight[t] * mean_returns[t] for t in tickers) >= target_return
    )

    model.variance = pyo.Objective(
        expr=sum(
            model.weight[i] * cov_matrix.loc[i, j] * model.weight[j]
            for i in tickers
            for j in tickers
        ),
        sense=pyo.minimize,
    )
    return model


def markowitz_min_variance(target_return: float) -> pd.Series:
    """Solve the min-variance model and return the resulting portfolio weights."""
    model = build_markowitz_model(target_return)
    tickers = list(model.tickers)

    # HiGHS: open-source QP/LP solver, no license required
    solver = pyo.SolverFactory("highs")
    result = solver.solve(model)
    pyo.assert_optimal_termination(result)

    weights = pd.Series({t: pyo.value(model.weight[t]) for t in tickers}, name="weight")
    # drop near-zero weights left over from solver numerical noise
    return weights[weights > 1e-6].sort_values(ascending=False)

### Visualization

In [4]:
def plot_allocation_by_stock(weights: pd.Series) -> go.Figure:
    weights = weights.sort_values(ascending=False)
    metadata = load_metadata().set_index("ticker")
    df = weights.rename("weight").rename_axis("ticker").reset_index()
    df = df.merge(metadata[["shortName", "sector"]], on="ticker", how="left")

    fig = px.bar(
        df,
        x="ticker",
        y="weight",
        color="weight",
        color_continuous_scale="Tealgrn",
        text="weight",
        hover_data={"ticker": False, "shortName": True, "sector": True, "weight": ":.1%"},
        title="Portfolio Allocation by Stock",
        template="plotly_white",
    )
    fig.update_traces(texttemplate="%{text:.1%}", textposition="outside")
    fig.update_layout(
        yaxis_tickformat=".0%",
        xaxis_title=None,
        yaxis_title="Weight",
        coloraxis_showscale=False,
        margin=dict(t=60),
    )
    fig.show()
    return fig


def plot_allocation_by_sector(weights: pd.Series) -> go.Figure:
    metadata = load_metadata().set_index("ticker")
    by_sector = weights.groupby(metadata.loc[weights.index, "sector"]).sum().sort_values(ascending=False)
    df = by_sector.rename("weight").rename_axis("sector").reset_index()

    fig = px.pie(
        df,
        names="sector",
        values="weight",
        title="Portfolio Allocation by Sector",
        hole=0.45,
        color_discrete_sequence=px.colors.qualitative.Prism,
        template="plotly_white",
    )
    fig.update_traces(textinfo="label+percent", pull=[0.03] * len(df), hoverinfo="skip", hovertemplate=None)
    fig.update_layout(showlegend=False, margin=dict(t=60))
    fig.show()
    return fig

## Excecution

In [5]:
prices = load_prices_with_metadata()
weights = markowitz_min_variance(target_return=0.20)
plot_allocation_by_stock(weights);
plot_allocation_by_sector(weights);

### Formal constraint checking with Z3

`build_markowitz_model` enforces that weights sum to 1, hit the target return, and lie
in `[0, 1]` — but it has **no per-stock concentration cap**, even though the
"no single stock over 10%" rule is a real requirement. Pyomo + HiGHS only ever hand
back *one* optimal point, so eyeballing that point is not a proof the rule holds
everywhere the model would allow the solver to go.

Z3 is an SMT solver: instead of optimizing, it searches for *any* assignment that
satisfies a set of constraints. Here we translate the pyomo model's actual variable
bounds and active constraints into Z3 (via `generate_standard_repn`, so this tracks the
model rather than a hand copy of it), then ask Z3 to satisfy those *plus* "some weight
exceeds 10%". If that combined query is SAT, we have a concrete counterexample showing
the pyomo model's feasible region violates the business rule — the rule was checked
against the program, not just against the one solution HiGHS happened to return.

In [6]:
def find_concentration_violation(model: pyo.ConcreteModel, max_weight: float = 0.10) -> None:
    """Ask Z3 whether the pyomo model's constraints allow any weight[t] > max_weight.

    Translates model.weight's bounds and every active pyo.Constraint into Z3 reals,
    then checks satisfiability against "some weight exceeds max_weight". SAT means
    the pyomo program's feasible region does not enforce the concentration cap, and
    the returned model is a concrete allocation that proves it.
    """
    tickers = list(model.tickers)
    z3_var = {id(model.weight[t]): z3.Real(t) for t in tickers}

    solver = z3.Solver()

    # Variable bounds, e.g. model.weight = pyo.Var(..., bounds=(0, 1))
    for t in tickers:
        v = model.weight[t]
        zv = z3_var[id(v)]
        if v.lb is not None:
            solver.add(zv >= float(v.lb))
        if v.ub is not None:
            solver.add(zv <= float(v.ub))

    # Every active linear constraint on the model (budget, target return, ...)
    for con in model.component_objects(pyo.Constraint, active=True):
        for index in con:
            c = con[index]
            repn = generate_standard_repn(c.body)
            terms = [coef * z3_var[id(v)] for coef, v in zip(repn.linear_coefs, repn.linear_vars)]
            body = (z3.Sum(terms) if terms else 0) + float(repn.constant)
            if c.equality:
                solver.add(body == float(c.lower))
            else:
                if c.has_lb():
                    solver.add(body >= float(c.lower))
                if c.has_ub():
                    solver.add(body <= float(c.upper))

    # The business rule the pyomo model above never encodes as a constraint
    solver.add(z3.Or([z3_var[id(model.weight[t])] > max_weight for t in tickers]))

    if solver.check() != z3.sat:
        print(f"UNSAT: the pyomo model's constraints already enforce the {max_weight:.0%} cap.")
        return

    z3_model = solver.model()
    allocation = {
        t: float(z3_model[z3_var[id(model.weight[t])]].as_fraction())
        for t in tickers
        if z3_model.eval(z3_var[id(model.weight[t])] != 0)
    }
    print(f"SAT: found a counterexample — pyomo's constraints permit a weight above {max_weight:.0%}.")
    print("Counterexample allocation (feasible under the pyomo model, violates the business rule):")
    for t, w in sorted(allocation.items(), key=lambda kv: -kv[1]):
        flag = "  <-- exceeds cap" if w > max_weight else ""
        print(f"  {t}: {w:.1%}{flag}")


find_concentration_violation(build_markowitz_model(target_return=0.20), max_weight=0.10)

SAT: found a counterexample — pyomo's constraints permit a weight above 10%.
Counterexample allocation (feasible under the pyomo model, violates the business rule):
  TXN: 10.6%  <-- exceeds cap
  PM: 10.6%  <-- exceeds cap
  INTU: 10.6%  <-- exceeds cap
  CAT: 10.6%  <-- exceeds cap
  VZ: 10.6%  <-- exceeds cap
  NOW: 10.6%  <-- exceeds cap
  AMGN: 10.6%  <-- exceeds cap
  ISRG: 10.6%  <-- exceeds cap
  QCOM: 10.6%  <-- exceeds cap
  IBM: 5.0%
